[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasonCiemielewski/SVEF-Drug-Rescue/blob/main/notebooks/model_creation.ipynb)

## Set Up

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import requests
import time
import re

# Detect Environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/BIFX546/'
    DATA_DIR = os.path.join(PROJECT_ROOT, 'data/demo/')
    sys.path.append(PROJECT_ROOT)
    !pip install -q matplotlib-venn
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    DATA_DIR = os.path.join(PROJECT_ROOT, 'data/demo/')
    sys.path.append(PROJECT_ROOT)

print(f"Project Root added to path: {PROJECT_ROOT}")
print(f"Data Directory: {DATA_DIR}")

Project Root added to path: c:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project
Data Directory: c:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project\data/demo/


In [2]:
# Run this inside the script file, not just the console

print(f"Is PROJECT_ROOT still here? {PROJECT_ROOT}")
print(f"Currently defined variables: {list(locals().keys())}")

Is PROJECT_ROOT still here? c:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project
Currently defined variables: ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'help', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'os', 'sys', 'pd', 'np', 'datetime', 'requests', 'time', 're', 'IN_COLAB', 'PROJECT_ROOT', 'DATA_DIR', '_i2']


In [2]:
INTERIM_DIR = os.path.join(PROJECT_ROOT, 'data', 'interim')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
RAW_DIR = os.path.join(PROJECT_ROOT, 'data','raw')

## **Filtering nct_id**

Trials are filtered to identify nct_id that have the following characteristics.  After each data file is filtered, a set of nct_id is created to subset the next datafile prior to further filtering  
**studies.txt**  
*overall_status*  
- COMPLETED
- UNKNOWN
- TERMINATED
- WITHDRAWN
- SUSPENDED
- ENROLLING_BY_INVITATION
- ACTIVE_NOT_RECRUITING

*study_type*  
- INTERVENTIONAL

(create nct_id_filter_set)

**design_groups.txt** 
*group_type*  
- EXPERIMENTAL

(create nct_id_filter_set2)

**interventions.txt**  
*intervention_type*  
- DRUG

(create nct_id_filter_set3)

### **studies.txt**

In [3]:
## set studies_features equal to the features to keep for subsetting the studies.txt RAW data
studies_features = ['nct_id', 'overall_status', 'phase', 'study_type', 'enrollment', 'enrollment_type', 'number_of_arms', 'why_stopped', 'is_fda_regulated_drug', 'is_fda_regulated_device']

## set status_list equal to overal_status to keep 
status_list = [
    'COMPLETED', 'UNKNOWN', 'TERMINATED', 'WITHDRAWN',
    'SUSPENDED', 'ENROLLING_BY_INVITATION', 'ACTIVE_NOT_RECRUITING'
]

In [4]:
# load studies.txt using only studies_features
studies_df = pd.read_csv(
    os.path.join(RAW_DIR, 'studies.txt'),
    sep = '|',
    usecols = studies_features
)

In [5]:
# Apply filters to studies_df
mask = (studies_df['study_type'] == 'INTERVENTIONAL') & (studies_df['overall_status'].isin(status_list))
studies_filtered_df = studies_df[mask].copy()

In [6]:
# verify filtering
print(f"Total trials in subset: {len(studies_filtered_df)}")
print("Unique statuses found:", studies_filtered_df['overall_status'].unique())

Total trials in subset: 374913
Unique statuses found: <ArrowStringArray>
[                'UNKNOWN',   'ACTIVE_NOT_RECRUITING',
               'COMPLETED',               'WITHDRAWN',
              'TERMINATED', 'ENROLLING_BY_INVITATION',
               'SUSPENDED']
Length: 7, dtype: str


In [7]:
## Create a set of the nct_id in studies_filtered_df for use in other RAW datasets
nct_id_filter_set = set(studies_filtered_df['nct_id'])
print(f"Filter set created with {len(nct_id_filter_set):,} unique IDs")

Filter set created with 374,913 unique IDs


In [8]:
#Extract studies_filtered_df column names into a list
studies_filtered_headers = studies_filtered_df.columns.tolist()

#Verify the list
print(f"Captured {len(studies_filtered_headers)} headers.")
print(studies_filtered_headers)

Captured 10 headers.
['nct_id', 'study_type', 'overall_status', 'phase', 'enrollment', 'enrollment_type', 'number_of_arms', 'why_stopped', 'is_fda_regulated_drug', 'is_fda_regulated_device']


In [9]:
# Remove mask, studies_df, studies_filtered_df
import gc

items_to_del = [
    'mask', 'studies_df', 'studies_filtered_df'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Success: mask was found and cleared from memory.
Success: studies_df was found and cleared from memory.
Success: studies_filtered_df was found and cleared from memory.


815

### **design_groups.txt**

In [10]:
# Create an empty list to store the pieces that match your IDs
design_chunks = []

# Pass the path directly into the loop
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_groups.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        low_memory=False):
    
    # Filter: Keep only rows where the nct_id is in your pre-defined set
    filtered_chunk = chunk[chunk['nct_id'].isin(nct_id_filter_set)]
    
    if not filtered_chunk.empty:
        design_chunks.append(filtered_chunk)

# Combine results
design_groups_filtered_df = pd.concat(design_chunks, axis=0).reset_index(drop=True)

In [11]:
design_groups_filtered_df['group_type'].isna().sum() # 0

np.int64(0)

In [12]:
# Apply filters to design_groups_filtered_df
mask = (design_groups_filtered_df['group_type'] == 'EXPERIMENTAL')
design_groups_filtered_df2 = design_groups_filtered_df[mask].copy()

In [13]:
# Verify that ONLY Experimental groups remain
print(design_groups_filtered_df2['group_type'].value_counts())

group_type
EXPERIMENTAL    430224
Name: count, dtype: int64


In [14]:
# 1. Create the refined set of NCT IDs
nct_id_filter_set2 = set(design_groups_filtered_df2['nct_id'])

# 2. Quick validation
print(f"Original subset count: {len(nct_id_filter_set):,}")
print(f"Refined 'Experimental' count: {len(nct_id_filter_set2):,}")
print(f"Trials filtered out: {len(nct_id_filter_set) - len(nct_id_filter_set2):,}")

Original subset count: 374,913
Refined 'Experimental' count: 286,303
Trials filtered out: 88,610


In [16]:
print(f"Length of unigue nct_id in design_groups_filtered_df", len(design_groups_filtered_df['nct_id'].unique()))
print(f"Length of nct_id in design_groups_filtered_df", len(design_groups_filtered_df['nct_id']))
print(f"Length of unique nct_id in design_groups_filtered_df2", len(design_groups_filtered_df2['nct_id'].unique()))
print(f"Length of nct_id in design_groups_filtered_df2", len(design_groups_filtered_df2['nct_id']))

Length of unigue nct_id in design_groups_filtered_df 351134
Length of nct_id in design_groups_filtered_df 758454
Length of unique nct_id in design_groups_filtered_df2 286303
Length of nct_id in design_groups_filtered_df2 430224


In [15]:
# Remove 'chunk', 'design_groups_filtered_df', 'design_groups_filtered_df2',
#        'design_chunks' 
import gc

items_to_del = [
    'chunk', 'design_groups_filtered_df', 'design_groups_filtered_df2',
    'design_chunks'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Success: chunk was found and cleared from memory.
Success: design_groups_filtered_df was found and cleared from memory.
Success: design_groups_filtered_df2 was found and cleared from memory.
Success: design_chunks was found and cleared from memory.


69

### **interventions.txt**

In [16]:
# The "Sieve" Approach
drug_interventions_chunks = []

# Intake 50,000 rows at a time
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'interventions.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['id', 'nct_id', 'intervention_type', 'name']):
    
    # Apply BOTH filters (Type and ID) while the chunk is on the "desk"
    mask = (chunk['intervention_type'].str.upper() == 'DRUG') & \
           (chunk['nct_id'].isin(nct_id_filter_set2))
    
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        drug_interventions_chunks.append(filtered_chunk)

# Assemble only the relevant data
filtered_interventions_df = pd.concat(drug_interventions_chunks, axis=0).reset_index(drop=True)

In [17]:
# 1. Create the refined set of NCT IDs
nct_id_filter_set3 = set(filtered_interventions_df['nct_id'])

# 2. Quick validation
print(f"Original subset count: {len(nct_id_filter_set2):,}")
print(f"Refined 'Experimental' count: {len(nct_id_filter_set3):,}")
print(f"Trials filtered out: {len(nct_id_filter_set2) - len(nct_id_filter_set3):,}")

Original subset count: 286,303
Refined 'Experimental' count: 128,186
Trials filtered out: 158,117


In [18]:
# Remove chunk, filtered_chunk, filtered_interventions_df,
#    mask, drug_interventions_chunks 
import gc

items_to_del = [
    'chunk', 'filtered_chunk', 'filtered_interventions_df',
    'mask', 'drug_interventions_chunks'
]

for item in items_to_del:
    # In a notebook, variables are stored in the globals() dictionary
    if item in globals():
        del globals()[item]
        print(f"Success: {item} was found and cleared from memory.")
    else:
        print(f"Notice: {item} was not found. No action needed.")

# Trigger the garbage collector once at the end to free up the RAM
gc.collect()

Success: chunk was found and cleared from memory.
Success: filtered_chunk was found and cleared from memory.
Success: filtered_interventions_df was found and cleared from memory.
Success: mask was found and cleared from memory.
Success: drug_interventions_chunks was found and cleared from memory.


49

## **Safety Features**

reported_events.txt data file.  

The features of interest to be kept in reported_events.txt is 'id', 'nct_id', 'result_group_id', 'event_type', 'subjects_affected', 
'subjects_at_risk', 'description', 'event_count', 'adverse_event_term', 'frequency_threshold', 'time_frame', 'ctgov_group_code'.

In [21]:
rep_event_features = [
    'id', 'nct_id', 'result_group_id', 'event_type', 'subjects_affected',
    'subjects_at_risk', 'description', 'event_count', 'adverse_event_term',
    'frequency_threshold', 'time_frame', 'ctgov_group_code'
]

The reported_events.txt file is very large and loading the entire file may cause a crash.  the reported_events.txt file is loaded in chunks and filtered for the desired nct_id. (nct_id_filter_set3)

In [22]:
# The "Sieve" Approach
rep_events_chunks = []

# Intake 50,000 rows at a time
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'reported_events.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=rep_event_features):
    
    # Apply BOTH filters (Type and ID) while the chunk is on the "desk"
    mask = (chunk['nct_id'].isin(nct_id_filter_set3))
    
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        rep_events_chunks.append(filtered_chunk)

# Assemble only the relevant data
filtered_rep_events_df = pd.concat(rep_events_chunks, axis=0).reset_index(drop=True)

In [23]:
# Create a binary feature: Did the investigator actually report the count?
# 1 = event_count has data
# 0 = event_count is NaN
filtered_rep_events_df['is_intensity_reported'] = filtered_rep_events_df['event_count'].notna().astype(int)

In [ ]:
filtered_rep_events_df.columns

In [ ]:
len(filtered_rep_events_df['nct_id'].unique())

##### **Arm Level Aggregation**

In [24]:
#Filter for 'serious' events to focus on SAE (Serious Adverse Events)
# We isolate serious events because they are the primary safety signals for drug rescue
sae_only_df = filtered_rep_events_df[filtered_rep_events_df['event_type'] == 'serious'].copy()

In [25]:
# 2. Perform the Aggregation at the Arm Level (nct_id + result_group_id)
# Note: Use .max() for subjects_at_risk because it is the same for all events in an arm
arm_level_safety = sae_only_df.groupby(['nct_id', 'result_group_id', 'ctgov_group_code']).agg(
    total_sae_occurrences=('event_count', 'sum'),      # Total times SAEs happened (Intensity numerator)
    sum_subjects_affected=('subjects_affected', 'sum'), # Sum of people affected per term (Note: may have overlaps)
    arm_subjects_at_risk=('subjects_at_risk', 'max'),  # Total population of the arm (Incidence denominator)
    intensity_reported_count=('is_intensity_reported', 'sum'),
    total_sae_rows=('id', 'count')
).reset_index()

$$SAE \text{ Intensity} = \frac{\sum \text{Total SAE Occurrences}}{\sum \text{Subjects Affected}}$$

In [26]:
# 3. Calculate Safety Features for the Random Forest
# Calculate Intensity: Average number of SAEs per affected subject
arm_level_safety['sae_intensity'] = (
    arm_level_safety['total_sae_occurrences'] / arm_level_safety['sum_subjects_affected']
).replace([np.inf, -np.inf], np.nan).fillna(0)

$$SAE\text{ Incidence Rate} = \min\left(1.0, \frac{\sum \text{Subjects Affected}}{\text{Arm Subjects at Risk}}\right)$$

In [27]:
# Calculate Incidence: What percentage of the arm population experienced a serious event
# We use .clip(upper=1.0) because summing subjects_affected across different terms can exceed 100% due to overlaps
arm_level_safety['sae_incidence_rate'] = (
    arm_level_safety['sum_subjects_affected'] / arm_level_safety['arm_subjects_at_risk']
).clip(upper=1.0).fillna(0)

In [28]:
# Calculate Data Quality: Percentage of rows where intensity was actually reported
arm_level_safety['pct_intensity_reported'] = (
    arm_level_safety['intensity_reported_count'] / arm_level_safety['total_sae_rows']
).fillna(0)

In [29]:
print(f"Aggregated {len(sae_only_df):,} event rows into {len(arm_level_safety):,} arm-level profiles.")

Aggregated 3,292,517 event rows into 78,107 arm-level profiles.


In [30]:
arm_level_safety.columns

Index(['nct_id', 'result_group_id', 'ctgov_group_code',
       'total_sae_occurrences', 'sum_subjects_affected',
       'arm_subjects_at_risk', 'intensity_reported_count', 'total_sae_rows',
       'sae_intensity', 'sae_incidence_rate', 'pct_intensity_reported'],
      dtype='str')

In [31]:
filtered_rep_events_df.columns

Index(['id', 'nct_id', 'result_group_id', 'ctgov_group_code', 'time_frame',
       'event_type', 'subjects_affected', 'subjects_at_risk', 'description',
       'event_count', 'adverse_event_term', 'frequency_threshold',
       'is_intensity_reported'],
      dtype='str')

##### **Mortality**

In [32]:
# Updated features for totals
totals_chunks = []
totals_features = ['nct_id', 'ctgov_group_code', 'event_type', 'subjects_affected', 'subjects_at_risk']

for chunk in pd.read_csv(os.path.join(RAW_DIR, 'reported_event_totals.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=totals_features):
    
    # Using existing filter nct_id_filter_set3
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        totals_chunks.append(filtered_chunk)

filtered_totals_df = pd.concat(totals_chunks, axis=0).reset_index(drop=True)

$$Mortality\ Rate_{arm} = \frac{\sum \text{Subjects\ Affected}_{(\text{event\_type} = \text{'deaths'})}}{\max(\text{Subjects\ at\ Risk}_{arm})}$$


In [33]:
# 1. Calculate mortality per arm using the correct 'deaths' label
# We use .strip() just in case there are hidden spaces in the strings
arm_mortality = filtered_totals_df[filtered_totals_df['event_type'].str.strip() == 'deaths'].groupby(['nct_id', 'ctgov_group_code']).agg(
    deaths=('subjects_affected', 'sum'),
    at_risk=('subjects_at_risk', 'max')
).reset_index()

# 2. Calculate the mortality rate feature
arm_mortality['mortality_rate'] = (arm_mortality['deaths'] / arm_mortality['at_risk']).fillna(0)

print(f"Success! Captured mortality data for {len(arm_mortality):,} experimental arms.")

Success! Captured mortality data for 110,684 experimental arms.


In [34]:
arm_mortality.describe()

,deaths,at_risk,mortality_rate
count,110684.000000,67571.000000,110684.000000
mean,4.332605,86.571651,0.073998
std,28.297907,1013.855677,0.208582
min,0.000000,0.000000,0.000000
25%,0.000000,7.000000,0.000000
50%,0.000000,21.000000,0.000000
75%,0.000000,60.000000,0.000000
max,1361.000000,242344.000000,1.000000


In [35]:
# 1. Check if the initial sieve actually caught anything
print(f"Total rows in filtered_totals_df: {len(filtered_totals_df):,}")

# 2. Check the exact labels used in the event_type column
if len(filtered_totals_df) > 0:
    print("Unique event types found:", filtered_totals_df['event_type'].unique())
else:
    print("WARNING: filtered_totals_df is empty. Check nct_id_filter_set3.")

# 3. Check a sample of nct_ids to see if they exist in the raw file
print("Sample of IDs we are looking for:", list(nct_id_filter_set3)[:5])

Total rows in filtered_totals_df: 332,052
Unique event types found: <ArrowStringArray>
['deaths', 'other', 'serious']
Length: 3, dtype: str
Sample of IDs we are looking for: ['NCT00564564', 'NCT03806231', 'NCT03719001', 'NCT01967524', 'NCT02397694']


In [36]:
# Join the Mortality data to your existing arm_level_safety table
# The join on both NCT_ID and CTGOV_GROUP_CODE prevents data collisions
final_arm_safety_profile = pd.merge(
    arm_level_safety, 
    arm_mortality[['nct_id', 'ctgov_group_code', 'deaths', 'mortality_rate']], 
    on=['nct_id', 'ctgov_group_code'], 
    how='left'
).fillna(0)

# Quick check of the final features
print(final_arm_safety_profile[['sae_incidence_rate', 'mortality_rate', 'sae_intensity']].head())

   sae_incidence_rate  mortality_rate  sae_intensity
0            0.522727             0.0       2.478261
1            0.526882             0.0       2.326531
2            0.709677             0.0       2.227273
3            0.560748             0.0       2.550000
4            0.574803             0.0       2.465753


### **Relational Bridge**

In [ ]:
# # The Sieve for the Bridge Table
# bridge_chunks = []
# # design_group_id matches the result_group_id from your events table
# bridge_features = ['nct_id', 'design_group_id', 'intervention_id']

# for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_group_interventions.txt'), 
#                         sep='|', 
#                         chunksize=50000, 
#                         usecols=bridge_features):
    
#     # Filter using your refined drug-trial set
#     mask = chunk['nct_id'].isin(nct_id_filter_set3)
#     filtered_chunk = chunk[mask]
    
#     if not filtered_chunk.empty:
#         bridge_chunks.append(filtered_chunk)

# filtered_bridge_df = pd.concat(bridge_chunks, axis=0).reset_index(drop=True)

In [ ]:
# # 1. Check Data Types (The #1 cause of empty joins)
# print("--- Data Type Check ---")
# print(f"Safety ID Type (result_group_id): {final_arm_safety_profile['result_group_id'].dtype}")
# print(f"Bridge ID Type (design_group_id): {filtered_bridge_df['design_group_id'].dtype}")

# # 2. Check for Overlap in NCT IDs
# safety_ncts = set(final_arm_safety_profile['nct_id'])
# bridge_ncts = set(filtered_bridge_df['nct_id'])
# overlap_ncts = safety_ncts.intersection(bridge_ncts)
# print(f"\n--- ID Overlap Check ---")
# print(f"NCT IDs in Safety: {len(safety_ncts):,}")
# print(f"NCT IDs in Bridge: {len(bridge_ncts):,}")
# print(f"NCT IDs appearing in BOTH: {len(overlap_ncts):,}")

# # 3. Look at a Sample from both sides for one overlapping NCT ID
# if len(overlap_ncts) > 0:
#     sample_nct = list(overlap_ncts)[0]
#     print(f"\n--- Sample for {sample_nct} ---")
#     print("Safety Side IDs:", final_arm_safety_profile[final_arm_safety_profile['nct_id'] == sample_nct]['result_group_id'].tolist())
#     print("Bridge Side IDs:", filtered_bridge_df[filtered_bridge_df['nct_id'] == sample_nct]['design_group_id'].tolist())

In [ ]:
# # 1. Get Result Titles (Links your Safety Metrics to a name)
# result_group_chunks = []
# for chunk in pd.read_csv(os.path.join(RAW_DIR, 'result_groups.txt'), 
#                         sep='|', 
#                         chunksize=50000, 
#                         usecols=['nct_id', 'id', 'title']):
#     # Use your pre-defined drug-trial filter
#     mask = chunk['nct_id'].isin(nct_id_filter_set3)
#     filtered_chunk = chunk[mask]
#     if not filtered_chunk.empty:
#         result_group_chunks.append(filtered_chunk)

# # Rename 'id' to 'result_group_id' to match your safety profile
# result_titles_df = pd.concat(result_group_chunks).rename(columns={'id': 'result_group_id', 'title': 'arm_title'})

In [ ]:
# # 2. Get Design Titles (Links your drugs to a name)
# design_group_chunks = []
# for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_groups.txt'), 
#                         sep='|', 
#                         chunksize=50000, 
#                         usecols=['nct_id', 'id', 'title']):
#     mask = chunk['nct_id'].isin(nct_id_filter_set3)
#     filtered_chunk = chunk[mask]
#     if not filtered_chunk.empty:
#         design_group_chunks.append(filtered_chunk)

# # Rename 'id' to 'design_group_id' to match your bridge table
# design_titles_df = pd.concat(design_group_chunks).rename(columns={'id': 'design_group_id', 'title': 'arm_title'})

In [ ]:
# # A. Build the "Rosetta Stone" by matching on NCT ID and Arm Title
# universal_bridge = pd.merge(
#     result_titles_df,
#     design_titles_df,
#     on=['nct_id', 'arm_title'],
#     how='inner'
# )

# # B. Link your safety profile to this new bridge
# safety_with_design_ids = pd.merge(
#     final_arm_safety_profile,
#     universal_bridge[['nct_id', 'result_group_id', 'design_group_id']],
#     on=['nct_id', 'result_group_id'],
#     how='inner'
# )

# # C. Finally, join to your Intervention IDs (The Drugs)
# # Using filtered_bridge_df from your Cell 82
# drug_safety_mapped = pd.merge(
#     safety_with_design_ids,
#     filtered_bridge_df[['nct_id', 'design_group_id', 'intervention_id']],
#     on=['nct_id', 'design_group_id'],
#     how='inner'
# )

# print(f"Success! Linked safety data for {drug_safety_mapped['intervention_id'].nunique():,} unique drugs.")

In [ ]:
# print(drug_safety_mapped.columns)

In [ ]:
# len(drug_safety_mapped)

In [ ]:
# # Save your progress!
# drug_safety_mapped.to_csv(os.path.join(INTERIM_DIR, 'drug_safety_mapped.csv'), index=False)

In [ ]:
# # Remove arm_level_safety, arm_mortality, chunk, design_titles_df, 
# ##     filtered_bridge_df, filtered_chunk, filtered_rep_events_df, 
# import gc

# items_to_del = [
#     'arm_level_safety', 'arm_mortality', 'chunk', 'design_titles_df',
#     'filtered_bridge_df', 'filtered_chunk', 'filtered_rep_events_df',
#     'filtered_totals_df', 'final_arm_safety_profile', 'mask', 'result_titles_df',
#     'sae_only_df', 'safety_with_design_ids', 'universal_bridge'
# ]

# for item in items_to_del:
#     # In a notebook, variables are stored in the globals() dictionary
#     if item in globals():
#         del globals()[item]
#         print(f"Success: {item} was found and cleared from memory.")
#     else:
#         print(f"Notice: {item} was not found. No action needed.")

# # Trigger the garbage collector once at the end to free up the RAM
# gc.collect()

In [41]:
# --- SECTION: RELATIONAL BRIDGE (THE ROSETTA STONE) ---

# 1. Load Prerequisite Design & Results Metadata
# We extract planned titles (Design) and reported titles (Results) simultaneously.
design_groups_list = []
result_titles_list = []
bridge_list = []
interventions_list = []

# A. Design Groups (Planned Arms)
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_groups.txt'), 
                        sep='|', chunksize=100000, 
                        usecols=['nct_id', 'id', 'title']):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    if mask.any():
        design_groups_list.append(chunk[mask])
design_groups_df = pd.concat(design_groups_list).rename(columns={'id': 'design_group_id', 'title': 'arm_title'})

# B. Result Groups (Reported Arms)
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'result_groups.txt'), 
                        sep='|', chunksize=100000, 
                        usecols=['nct_id', 'id', 'title']):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    if mask.any():
        result_titles_list.append(chunk[mask])
result_titles_df = pd.concat(result_titles_list).rename(columns={'id': 'result_group_id', 'title': 'arm_title'})

# C. Intervention Bridge (Arm -> Drug ID Mapping)
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'design_group_interventions.txt'), 
                        sep='|', chunksize=100000, 
                        usecols=['nct_id', 'design_group_id', 'intervention_id']):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    if mask.any():
        bridge_list.append(chunk[mask])
filtered_bridge_df = pd.concat(bridge_list)

# D. Intervention Names (Drug Nomenclature)
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'interventions.txt'), 
                        sep='|', chunksize=100000, 
                        usecols=['id', 'nct_id', 'intervention_type', 'name']):
    mask = (chunk['intervention_type'].str.upper() == 'DRUG') & (chunk['nct_id'].isin(nct_id_filter_set3))
    if mask.any():
        interventions_list.append(chunk[mask])
filtered_interventions_df = pd.concat(interventions_list).rename(columns={'id': 'intervention_id'})

# 2. Attach Titles to Safety Data
# This resolves the KeyError by ensuring final_arm_safety_profile HAS an 'arm_title' column.
final_arm_safety_profile = pd.merge(
    final_arm_safety_profile,
    result_titles_df[['nct_id', 'result_group_id', 'arm_title']],
    on=['nct_id', 'result_group_id'],
    how='inner'
)

# 3. String Standardization for Composite Join
# We use lowercase and strip whitespace to ensure the 'Handshake' is precise.
for df in [final_arm_safety_profile, design_groups_df]:
    df['nct_id'] = df['nct_id'].astype(str).str.strip()
    df['arm_title'] = df['arm_title'].astype(str).str.strip().str.lower()

# 4. The Master Handshake (Composite Join)
# Linking Safety (Results Domain) to planned architecture (Design Domain) via Title.
master_ml_df = pd.merge(
    final_arm_safety_profile,
    design_groups_df,
    on=['nct_id', 'arm_title'],
    how='inner'
)

# 5. Final Domain Integration
# Step A: Link to specific Drug IDs
master_ml_df = pd.merge(master_ml_df, filtered_bridge_df, on=['nct_id', 'design_group_id'], how='inner')
# Step B: Link to Human-Readable Drug Names
master_ml_df = pd.merge(master_ml_df, filtered_interventions_df[['intervention_id', 'name']], on='intervention_id', how='inner')

# 6. Pipeline Memory Management & Audit
del result_titles_df, design_groups_df, filtered_bridge_df, filtered_interventions_df
import gc
gc.collect()

print(f"--- Relational Bridge Final Audit ---")
print(f"Integrated Records: {len(master_ml_df):,}")
print(f"Unique Interventions: {master_ml_df['intervention_id'].nunique():,}")

--- Relational Bridge Final Audit ---
Integrated Records: 50,368
Unique Interventions: 34,215


In [42]:
master_ml_df.columns

Index(['nct_id', 'result_group_id', 'ctgov_group_code',
       'total_sae_occurrences', 'sum_subjects_affected',
       'arm_subjects_at_risk', 'intensity_reported_count', 'total_sae_rows',
       'sae_intensity', 'sae_incidence_rate', 'pct_intensity_reported',
       'deaths', 'mortality_rate', 'arm_title_x', 'arm_title_y', 'arm_title',
       'design_group_id', 'intervention_id', 'name'],
      dtype='str')

## **Efficacy Features**

**outcomes.txt** 
*id*
*nct_id* 
*outcome_type*  
- 'primary'

**outcome_analyses.txt**  
 *nct_id*   
 *outcome_id*   
 *non_inferiority_type*   
 *p_value*

In [43]:
# 1. Identify Primary Outcomes
primary_outcome_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'outcomes.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['id', 'nct_id', 'outcome_type']):
    # Filter for your drug trials and 'primary' outcome types
    mask = (chunk['nct_id'].isin(nct_id_filter_set3)) & \
           (chunk['outcome_type'].str.lower() == 'primary')
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        primary_outcome_chunks.append(filtered_chunk)

primary_outcomes_df = pd.concat(primary_outcome_chunks).rename(columns={'id': 'outcome_id'})

In [ ]:
primary_outcomes_df.columns

In [44]:
# Updated features to include 'id' so we can link to specific arms
p_val_features = ['id', 'nct_id', 'outcome_id', 'non_inferiority_type', 'p_value']

p_val_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'outcome_analyses.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=p_val_features):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    filtered_chunk = chunk[mask]
    if not filtered_chunk.empty:
        p_val_chunks.append(filtered_chunk)

raw_p_values_df = pd.concat(p_val_chunks).rename(columns={'id': 'outcome_analysis_id'})

In [45]:
# 1. Load the link between analyses and result groups
analysis_groups_chunks = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'outcome_analysis_groups.txt'), 
                        sep='|', 
                        chunksize=100000, 
                        usecols=['nct_id', 'outcome_analysis_id', 'result_group_id']):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    if mask.any():
        analysis_groups_chunks.append(chunk[mask])

analysis_groups_bridge = pd.concat(analysis_groups_chunks)

# 2. Add 'result_group_id' to your raw p-values
raw_p_values_df = pd.merge(
    raw_p_values_df,
    analysis_groups_bridge[['outcome_analysis_id', 'result_group_id']],
    on='outcome_analysis_id',
    how='inner'
)

print(f"Success! Linked {len(raw_p_values_df):,} p-values to specific treatment arms.")

Success! Linked 390,808 p-values to specific treatment arms.


In [46]:
#We join the p-values to the outcome type using the outcome_id
efficacy_bridge = pd.merge(
    raw_p_values_df,
    primary_outcomes_df[['nct_id', 'outcome_id', 'outcome_type']],
    on=['nct_id', 'outcome_id'],
    how='inner'
)

print(f"Success! 'efficacy_bridge' created with {len(efficacy_bridge):,} analysis rows.")

Success! 'efficacy_bridge' created with 81,018 analysis rows.


In [47]:
# Join to Primary Outcomes (using your existing primary_outcomes_df)
# 1. Clean the p-values by forcing the column to string type first
# This prevents the AttributeError when encountering floats or NaNs
efficacy_bridge['p_value_clean'] = pd.to_numeric(
    efficacy_bridge['p_value'].astype(str).str.replace(r'[<>=\s]', '', regex=True), 
    errors='coerce'
)

# 2. Standardize the non_inferiority_type
# We also use .astype(str) here because NaNs in this column will cause the same error
efficacy_bridge['non_inferiority_type'] = (
    efficacy_bridge['non_inferiority_type']
    .astype(str)
    .str.title()
    .str.strip()
)

# Standardize the types (to handle case sensitivity)
efficacy_bridge['non_inferiority_type'] = efficacy_bridge['non_inferiority_type'].str.title().str.strip()

# DEFINE THE SIGNAL:
# A "Superiority Failure" is our prime rescue candidate (Stalled IP)
def label_failure(row):
    if pd.isna(row['p_value_clean']):
        return "Unknown"
    if row['non_inferiority_type'] == 'Superiority':
        return 1 if row['p_value_clean'] > 0.05 else 0
    elif row['non_inferiority_type'] == 'Non-Inferiority':
        # If a non-inferiority trial fails (p > 0.05), it's a Performance Failure
        return 2 if row['p_value_clean'] > 0.05 else 0
    return 0

efficacy_bridge['efficacy_failure_label'] = efficacy_bridge.apply(label_failure, axis=1)

In [ ]:
efficacy_bridge.columns

In [ ]:
efficacy_bridge.head()

In [48]:
# Save your progress!
efficacy_bridge.to_csv(os.path.join(INTERIM_DIR, 'efficacy_bridge.csv'), index=False)

### **efficacy_bridge**

In [52]:
efficacy_bridge.columns

Index(['outcome_analysis_id', 'nct_id', 'outcome_id', 'non_inferiority_type',
       'p_value', 'result_group_id', 'outcome_type', 'p_value_clean',
       'efficacy_failure_label'],
      dtype='str')

In [49]:
# Loop directly through the existing dataframe
for col in efficacy_bridge.columns:
    print(f"--- Column: {col} ---")
    
    # Access the column directly from the main dataframe
    print(f"Missing data: {efficacy_bridge[col].isna().sum():,}")
    print(f"With data:    {efficacy_bridge[col].notna().sum():,}")
    print(f"Total length: {len(efficacy_bridge):,}")
    
    unique_vals = efficacy_bridge[col].nunique()
    print(f"Unique values: {unique_vals:,}")
    print("\n")

--- Column: outcome_analysis_id ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique values: 41,040


--- Column: nct_id ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique values: 13,712


--- Column: outcome_id ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique values: 21,936


--- Column: non_inferiority_type ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique values: 8


--- Column: p_value ---
Missing data: 23,285
With data:    57,733
Total length: 81,018
Unique values: 5,274


--- Column: result_group_id ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique values: 53,920


--- Column: outcome_type ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique values: 1


--- Column: p_value_clean ---
Missing data: 23,285
With data:    57,733
Total length: 81,018
Unique values: 5,274


--- Column: efficacy_failure_label ---
Missing data: 0
With data:    81,018
Total length: 81,018
Unique valu

In [50]:
# Provides a summary of all missing values in one small table
print(efficacy_bridge.isna().sum())

outcome_analysis_id           0
nct_id                        0
outcome_id                    0
non_inferiority_type          0
p_value                   23285
result_group_id               0
outcome_type                  0
p_value_clean             23285
efficacy_failure_label        0
dtype: int64


In [51]:
# 1. Inspect the distribution of Efficacy Labels
print("--- Efficacy Label Distribution (Arm Level) ---")
print(efficacy_bridge['efficacy_failure_label'].value_counts())

# 2. Calculate percentages to see the "Rescue Opportunity"
print("\n--- Percentage Breakdown ---")
print(efficacy_bridge['efficacy_failure_label'].value_counts(normalize=True) * 100)

# 3. Optional: Map the labels back to a readable string for a quick check
label_map = {0: "Success (p <= 0.05)", 1: "Stalled IP (Superiority Failure)", 2: "Performance Failure (Inferiority)", "Unknown": "Data Missing"}
print("\n--- Human-Readable Summary ---")
print(efficacy_bridge['efficacy_failure_label'].map(label_map).value_counts())

--- Efficacy Label Distribution (Arm Level) ---
efficacy_failure_label
0          46433
Unknown    23285
1          11300
Name: count, dtype: int64

--- Percentage Breakdown ---
efficacy_failure_label
0          57.311955
Unknown    28.740527
1          13.947518
Name: proportion, dtype: float64

--- Human-Readable Summary ---
efficacy_failure_label
Success (p <= 0.05)                 46433
Data Missing                        23285
Stalled IP (Superiority Failure)    11300
Name: count, dtype: int64


In [53]:
def clean_drug_name(drug_name):
    """
    Strips clinical trial prefixes, doses, and salt forms from drug names to 
    improve PubChem match rates.
    
    Args:
        drug_name (str): The raw intervention name.
        
    Returns:
        str: The cleaned drug name.
    """
    raw_name = str(drug_name)
    clean = re.sub(r'^(?:comparator|arm \d+|arm|group|active|placebo|sham|standard of care|vehicle|regimen)\s*[:\-]\s*', '', raw_name, flags=re.IGNORECASE)
    clean = re.sub(r'\[.*?\]', '', clean)
    clean = re.sub(r'\(.*?\)', '', clean)
    clean = clean.split(',')[0].split(';')[0].strip()
    
    dose_pattern = r'\b(?:\d+ ?mg|\d+ ?g|\d+ ?mcg|\d+ ?u/kg|iv|intravenous|oral|tablets?|capsules?|active drug|matching|hydrochloride|sodium|salt|ointment|gel|solution|capsule|product|treatment|arm|preceding|study|phase|forming|phosphate|besylate|acetate|fumarate|maleate|succinate|tartrate|citrate|mesylate)\b'
    clean = re.sub(dose_pattern, '', clean, flags=re.IGNORECASE).strip()
    clean = re.sub(r'\d*\.?\d*%', '', clean).strip()
    clean = re.sub(r'\s+\d+\.?\d*$', '', clean).strip()
    clean = re.sub(r'\s+', ' ', clean).strip()
    clean = re.sub(r'^[^a-zA-Z0-9]+|[^a-zA-Z0-9]+$', '', clean).strip()
    
    if not clean or len(clean) < 2 or 'placebo' in clean.lower(): 
        if re.match(r'^\d+ ?mg|tablet|capsule', raw_name, re.I) or 'placebo' in raw_name.lower():
            return ""
        return raw_name.split(',')[0].strip()
    return clean

## Joining drug_safety_mapped and efficacy_bridge

In [54]:
# The "Sieve" Approach
drug_interventions_chunks = []

# Intake 50,000 rows at a time
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'interventions.txt'), 
                        sep='|', 
                        chunksize=50000, 
                        usecols=['id', 'nct_id', 'intervention_type', 'name']):
    
    # Apply BOTH filters (Type and ID) while the chunk is on the "desk"
    mask = (chunk['intervention_type'].str.upper() == 'DRUG') & \
           (chunk['nct_id'].isin(nct_id_filter_set2))
    
    filtered_chunk = chunk[mask]
    
    if not filtered_chunk.empty:
        drug_interventions_chunks.append(filtered_chunk)

# Assemble only the relevant data
filtered_interventions_df = pd.concat(drug_interventions_chunks, axis=0).reset_index(drop=True)

In [55]:
# 1. Prepare Arm-Level Efficacy
# We group by BOTH nct_id and result_group_id to keep the focus on the specific drug arm
arm_efficacy = efficacy_bridge.groupby(['nct_id', 'result_group_id']).agg(
    arm_p_value=('p_value_clean', 'min'),
    is_superiority=('non_inferiority_type', lambda x: 1 if 'Superiority' in x.values else 0)
).reset_index()

# 2. Define the Target Label for each specific arm
arm_efficacy['is_arm_failure'] = (arm_efficacy['arm_p_value'] > 0.05).astype(int)

print(f"Efficacy labels created for {len(arm_efficacy):,} unique intervention arms.")

Efficacy labels created for 53,920 unique intervention arms.


In [57]:
# --- SECTION: EFFICACY INTEGRATION ---

# 1. Synchronize Data Types Across Both Dataframes
# This prevents ValueErrors by ensuring both sides of the join use strings.
for df in [master_ml_df, arm_efficacy]:
    for col in ['nct_id', 'result_group_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

# 2. Integrate Efficacy Labels
# We perform a 'left' join to append efficacy outcomes to our drug-mapped arms.
# We join on NCT ID and result_group_id for maximum precision.
master_ml_df = pd.merge(
    master_ml_df,
    arm_efficacy[['nct_id', 'result_group_id', 'is_arm_failure']],
    on=['nct_id', 'result_group_id'],
    how='left'
)

# 3. Final Labeling & Clinical Cleaning
# Arms with missing efficacy results are conservatively labeled as failures (1).
master_ml_df['is_arm_failure'] = master_ml_df['is_arm_failure'].fillna(1).astype(int)

# Use the clean_drug_name function to standardize drug nomenclature
if 'name' in master_ml_df.columns:
    master_ml_df['clean_name'] = master_ml_df['name'].apply(clean_drug_name)

# 4. Pipeline Finalization Audit
print(f"✅ Efficacy Integration Complete! Master Records: {len(master_ml_df):,}")
print("\n--- Arm Efficacy Distribution ---")
print(master_ml_df['is_arm_failure'].value_counts())

✅ Efficacy Integration Complete! Master Records: 50,368

--- Arm Efficacy Distribution ---
is_arm_failure
1    50368
Name: count, dtype: int64


In [58]:
# --- DIAGNOSTIC: WHY IS THE JOIN EMPTY? ---
print("--- ID Format Comparison ---")
print(f"Master DF Sample IDs: {master_ml_df['result_group_id'].head(3).tolist()}")
print(f"Efficacy DF Sample IDs: {arm_efficacy['result_group_id'].head(3).tolist()}")

# Check for NCT overlap specifically in the efficacy table
efficacy_ncts = set(arm_efficacy['nct_id'])
master_ncts = set(master_ml_df['nct_id'])
overlap = master_ncts.intersection(efficacy_ncts)

print(f"\n--- NCT Overlap Check ---")
print(f"NCTs in Master: {len(master_ncts):,}")
print(f"NCTs in Efficacy: {len(efficacy_ncts):,}")
print(f"NCTs appearing in BOTH: {len(overlap):,}")

--- ID Format Comparison ---
Master DF Sample IDs: ['684449868', '684449869', '684449870']
Efficacy DF Sample IDs: ['684055253', '684055254', '684055257']

--- NCT Overlap Check ---
NCTs in Master: 16,225
NCTs in Efficacy: 13,712
NCTs appearing in BOTH: 5,998


In [59]:
# --- SECTION: EFFICACY INTEGRATION & MASTER DATASET CREATION ---

# 1. Load Raw Efficacy Data (Outcome Results)
# We re-process this here to ensure it uses the EXACT same NCT filter as your safety data.
efficacy_list = []
# These columns are the standard for identifying trial failure in AACT
outcome_cols = ['nct_id', 'result_group_id', 'non_inferiority_type', 'p_value', 'param_type']

for chunk in pd.read_csv(os.path.join(RAW_DIR, 'outcome_results.txt'), 
                        sep='|', chunksize=100000, 
                        usecols=outcome_cols):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    if mask.any():
        efficacy_list.append(chunk[mask])

raw_efficacy_df = pd.concat(efficacy_list)

# 2. Define the Efficacy Failure Label (is_arm_failure)
# Logic: A trial arm is a 'Success' (0) if it has a p-value <= 0.05.
# Otherwise, it is a 'Failure' (1).
raw_efficacy_df['p_value'] = pd.to_numeric(raw_efficacy_df['p_value'], errors='coerce')
raw_efficacy_df['is_arm_failure'] = np.where(raw_efficacy_df['p_value'] <= 0.05, 0, 1)

# 3. Consolidate to Arm-Level Failure
# Since one arm can have multiple outcomes, we take the 'best' result (the minimum failure label)
arm_efficacy_labels = raw_efficacy_df.groupby(['nct_id', 'result_group_id'])['is_arm_failure'].min().reset_index()

# 4. Standardize IDs for the Master Handshake
# This resolves the 'int64 vs str' and '123.0 vs 123' artifacts simultaneously.
for df in [master_ml_df, arm_efficacy_labels]:
    for col in ['nct_id', 'result_group_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.split('.').str[0].str.strip()

# 5. The Master Join
# We join efficacy labels onto our drug-mapped safety master.
final_master_df = pd.merge(
    master_ml_df,
    arm_efficacy_labels,
    on=['nct_id', 'result_group_id'],
    how='left'
)

# 6. Final Labeling & Cleanup
# We conservatively assume any arm missing an efficacy record is a failure (1).
final_master_df['is_arm_failure'] = final_master_df['is_arm_failure'].fillna(1).astype(int)

# Apply nomenclature cleaning for the May 2026 presentation
if 'name' in final_master_df.columns:
    final_master_df['clean_name'] = final_master_df['name'].apply(clean_drug_name)

# 7. Memory Management & Audit
del raw_efficacy_df, arm_efficacy_labels
import gc
gc.collect()

print(f"✅ Master Integration Complete! Integrated Records: {len(final_master_df):,}")
print(f"Trials with available Efficacy Data: {final_master_df[final_master_df['is_arm_failure'] != 1].nct_id.nunique():,}")
print("\n--- Arm Efficacy Distribution (0=Success, 1=Failure) ---")
print(final_master_df['is_arm_failure'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Ownwer\\Documents\\BIFX\\BIFX546\\Final_Project\\data\\raw\\outcome_results.txt'

In [ ]:
import os
import pandas as pd

# 1. RE-LOAD THE TITLE MAP (The Rosetta Stone)
# This provides the link between numeric IDs and human-readable names
result_titles_list = []
for chunk in pd.read_csv(os.path.join(RAW_DIR, 'result_groups.txt'), 
                        sep='|', 
                        chunksize=100000, 
                        usecols=['nct_id', 'id', 'title']):
    mask = chunk['nct_id'].isin(nct_id_filter_set3)
    if mask.any():
        result_titles_list.append(chunk[mask])

result_titles_df = pd.concat(result_titles_list).rename(columns={'id': 'result_group_id', 'title': 'arm_title'})

# Synchronize ID types to ensure perfect string matching
for df in [result_titles_df, drug_safety_mapped, arm_efficacy, filtered_interventions_df]:
    for col in ['nct_id', 'result_group_id', 'intervention_id', 'id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.split('.').str[0].str.strip()

# 2. PREPARE SAFETY SIDE (Add Titles & Filter for Drugs)
# This ensures placebos/devices are purged and every drug arm has its title
drug_safety_with_titles = pd.merge(
    drug_safety_mapped,
    result_titles_df[['nct_id', 'result_group_id', 'arm_title']],
    on=['nct_id', 'result_group_id'],
    how='inner'
)
drug_safety_with_titles = pd.merge(
    drug_safety_with_titles,
    filtered_interventions_df[['id', 'name']].rename(columns={'id': 'intervention_id'}),
    on='intervention_id',
    how='inner'
)

# 3. PREPARE EFFICACY SIDE (Add Titles)
efficacy_with_titles = pd.merge(
    arm_efficacy,
    result_titles_df[['nct_id', 'result_group_id', 'arm_title']],
    on=['nct_id', 'result_group_id'],
    how='inner'
)

# 4. THE MASTER HANDSHAKE (Join on NCT ID + Arm Title)
# We join on the title to bypass AACT's surrogate key volatility
master_ml_df = pd.merge(
    drug_safety_with_titles,
    efficacy_with_titles.drop(columns=['result_group_id']), 
    on=['nct_id', 'arm_title'],
    how='left'
)

# 5. FINAL LABELING & CLEANING
master_ml_df['is_arm_failure'] = master_ml_df['is_arm_failure'].fillna(1).astype(int)
master_ml_df['clean_name'] = master_ml_df['name'].apply(clean_drug_name)

print(f"Master Integration Complete! Final Dataset Size: {len(master_ml_df):,}")
print("\n--- Arm Efficacy Distribution ---")
print(master_ml_df['is_arm_failure'].value_counts())

In [ ]:
# 1. Load only the metadata columns you need
# This keeps the memory footprint small
study_metadata = pd.read_csv(
    os.path.join(RAW_DIR, 'studies.txt'),
    sep='|',
    usecols=[
        'nct_id', 'overall_status', 'phase', 'study_type',
        'number_of_arms', 'why_stopped', 'is_fda_regulated_drug'],
    low_memory=False
)

# 2. Merge these features onto your Master ML Dataframe
# We use 'left' join to ensure every row in master_ml_df is preserved
master_ml_df = pd.merge(
    master_ml_df, 
    study_metadata, 
    on='nct_id', 
    how='left'
)

# 3. Quick check of the new columns
print(f"Features added! New shape: {master_ml_df.shape}")
print(master_ml_df[['nct_id', 'overall_status', 'phase', 'why_stopped']].head())

In [ ]:
# Save your progress!
master_ml_df.to_csv(os.path.join(INTERIM_DIR, 'master_ml_df.csv'), index=False)

In [ ]:
master_ml_df['phase'].value_counts()

In [ ]:
master_ml_df['overall_status'].value_counts()

In [ ]:
master_ml_df['is_arm_failure'].value_counts()

In [ ]:
# 1. Calculate basic transformation metrics
changes_mask = master_ml_df['name'] != master_ml_df['clean_name']
total_changes = changes_mask.sum()
percent_changed = (total_changes / len(master_ml_df)) * 100

# 2. Identify Potential "Edge Cases"
# Empty names (indicates a name that was entirely dose/noise)
empty_names = master_ml_df[master_ml_df['clean_name'] == '']
# Very short names (often codes like 'BMS' or errors)
short_names = master_ml_df[master_ml_df['clean_name'].str.len().between(1, 2, inclusive='both')]

print(f"--- Cleaning Audit Summary ---")
print(f"Total Rows Analyzed:      {len(master_ml_df):,}")
print(f"Names Transformed:        {total_changes:,} ({percent_changed:.1f}%)")
print(f"Names Resulting in Empty: {len(empty_names):,}")
print(f"Names < 3 Characters:     {len(short_names['clean_name'].unique()):,}")

# 3. Create a side-by-side comparison object of 100 changed names
# We use .drop_duplicates() to see unique transformations rather than repeated rows
transformation_sample = master_ml_df[changes_mask][['name', 'clean_name']].drop_duplicates().head(100)

# 4. Display the sample
print("\n--- Side-by-Side Comparison (First 100 Unique Changes) ---")
pd.set_option('display.max_colwidth', None)
print(transformation_sample)

In [ ]:
# 1. Remove rows where the clean_name is empty
# These represent the Placebos and shams identified during your audit
master_ml_df = master_ml_df[master_ml_df['clean_name'] != ''].copy()

# 2. Remove any remaining rows that are explicitly 'Placebo' (case insensitive)
master_ml_df = master_ml_df[~master_ml_df['clean_name'].str.lower().str.contains('placebo', na=False)]

# 3. Final count before API enrichment
print(f"Noise Purged! {len(master_ml_df):,} valid drug intervention arms remaining.")
print(f"Unique drugs to be queried: {master_ml_df['clean_name'].nunique():,}")

In [ ]:
master_ml_df['name'].isna().sum()

In [ ]:
master_ml_df['clean_name'].isna().sum()

In [ ]:
import requests
import time
import pandas as pd
from tqdm import tqdm

def test_selection_logic(drug_name):
    """Fetches top 5 results and compares the First Result vs. Our Best Match."""
    if not drug_name:
        return None
    
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{drug_name}/property/MolecularWeight,XLogP,Complexity/JSON?list_return=top5"
    
    try:
        response = requests.get(url, timeout=8)
        if response.status_code == 200:
            props = response.json()['PropertyTable']['Properties']
            
            # Naive choice (First result)
            first_match = props[0]
            
            # Heuristic choice (Sorted)
            sorted_matches = sorted(
                props,
                key=lambda x: (
                    x.get('XLogP') is None,     # Completeness first
                    x.get('MolecularWeight', 9999), # Lowest MW second
                    x.get('Complexity', 9999)    # Lowest Complexity third
                )
            )
            best_match = sorted_matches[0]
            
            return {
                'drug': drug_name,
                'total_results': len(props),
                'first_mw': first_match.get('MolecularWeight'),
                'best_mw': best_match.get('MolecularWeight'),
                'chose_different': first_match.get('CID') != best_match.get('CID')
            }
    except:
        pass
    return None

# 1. Sample 100 unique drugs for the test
test_subset = master_ml_df['clean_name'].unique()[:100]
print(f"Testing selection logic on {len(test_subset)} molecules...")

# 2. Run the comparison loop
test_results = []
for drug in tqdm(test_subset):
    res = test_selection_logic(drug)
    if res:
        test_results.append(res)
    time.sleep(0.2)

# 3. Analyze the impact
test_df = pd.DataFrame(test_results)
impact_df = test_df[test_df['chose_different'] == True]

print(f"\n--- Test Results ---")
print(f"Total Successful Queries: {len(test_df)}")
print(f"Times Logic Chose a Different CID: {len(impact_df)}")

if len(impact_df) > 0:
    print("\n--- Examples of the Logic 'Correcting' the Selection ---")
    print(impact_df[['drug', 'first_mw', 'best_mw']].head(10))

In [ ]:
import requests
from requests.utils import quote
import pandas as pd

# Let's pick 10 unique drugs from your data
test_list = master_ml_df['clean_name'].unique()[:10]

print(f"{'Original Name':<30} | {'Decision Logic'}")
print("-" * 70)

for drug in test_list:
    # 1. Translate the name for the web
    safe_name = quote(drug)
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{safe_name}/property/MolecularWeight,XLogP/JSON?list_return=top5"
    
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            results = response.json()['PropertyTable']['Properties']
            
            # 2. See how many results were found
            count = len(results)
            
            # 3. Apply the 'Lowest Weight' rule
            best_match = sorted(results, key=lambda x: x.get('MolecularWeight', 9999))[0]
            
            print(f"{drug:<30} | Found {count} versions. Selected MW: {best_match['MolecularWeight']}")
        else:
            print(f"{drug:<30} | Not found in PubChem.")
    except:
        print(f"{drug:<30} | Connection Error.")

In [ ]:
import requests
from requests.utils import quote

# Test just three common drugs to see why they are failing
test_list = ["Prednisone", "Etoposide", "Doxorubicin"]

print(f"{'Drug Name':<15} | {'Step 1 (Find ID)':<15} | {'Step 2 (Get Data)'}")
print("-" * 70)

for drug in test_list:
    # 1. Translate name for the web
    safe_name = quote(drug)
    
    # STEP 1: Just find the ID (CID)
    cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{safe_name}/cids/JSON"
    
    try:
        cid_response = requests.get(cid_url, timeout=10)
        
        if cid_response.status_code == 200:
            cid = cid_response.json()['IdentifierList']['CID'][0]
            
            # STEP 2: Now that we have a confirmed ID, get the properties
            prop_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/MolecularWeight,XLogP/JSON"
            prop_response = requests.get(prop_url, timeout=10)
            
            if prop_response.status_code == 200:
                data = prop_response.json()['PropertyTable']['Properties'][0]
                mw = data.get('MolecularWeight', 'N/A')
                print(f"{drug:<15} | Found CID: {cid:<8} | Success! MW: {mw}")
            else:
                print(f"{drug:<15} | Found CID: {cid:<8} | Property Request Failed (Code {prop_response.status_code})")
        else:
            print(f"{drug:<15} | CID Not Found   | URL tried: {cid_url}")
            
    except Exception as e:
        print(f"{drug:<15} | Error: {str(e)}")

In [ ]:
import requests
import time
from tqdm import tqdm
from requests.utils import quote

def get_molecular_data(drug_name):
    """
    Expert Two-Step Retrieval: 
    1. Finds all IDs for a name.
    2. Fetches data and selects the parent molecule (lowest MW).
    """
    if not drug_name or len(str(drug_name)) < 3:
        return None
    
    # STEP 1: Find all IDs (CIDs) associated with the name
    safe_name = quote(str(drug_name))
    cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{safe_name}/cids/JSON?max_records=5"
    
    try:
        cid_res = requests.get(cid_url, timeout=10)
        if cid_res.status_code != 200:
            return None
        
        cids = cid_res.json().get('IdentifierList', {}).get('CID', [])
        
        # STEP 2: Fetch properties for those specific CIDs
        cid_str = ",".join(map(str, cids))
        prop_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_str}/property/MolecularWeight,XLogP,CanonicalSMILES/JSON"
        
        prop_res = requests.get(prop_url, timeout=10)
        if prop_res.status_code == 200:
            props = prop_res.json()['PropertyTable']['Properties']
            
            # RULE: Sort by lowest Molecular Weight (to favor parent molecule over salts)
            # and prefer records that have XLogP data.
            best_match = sorted(
                props,
                key=lambda x: (x.get('XLogP') is None, x.get('MolecularWeight', 9999))
            )[0]
            
            return best_match
    except:
        pass
    return None

# --- EXECUTION ---

# 1. Get your unique drug list
unique_drugs = master_ml_df['clean_name'].unique()
results_cache = {}

print(f"Starting enrichment for {len(unique_drugs):,} unique drugs...")

# 2. Run the loop with a slight delay to respect PubChem's servers
for drug in tqdm(unique_drugs):
    data = get_molecular_data(drug)
    if data:
        results_cache[drug] = data
    time.sleep(0.2) 

# 3. Map the data back to your main table
master_ml_df['molecular_weight'] = master_ml_df['clean_name'].map(lambda x: results_cache.get(x, {}).get('MolecularWeight'))
master_ml_df['xlogp'] = master_ml_df['clean_name'].map(lambda x: results_cache.get(x, {}).get('XLogP'))
master_ml_df['smiles'] = master_ml_df['clean_name'].map(lambda x: results_cache.get(x, {}).get('CanonicalSMILES'))

print(f"\nEnrichment Complete! Found data for {len(results_cache):,} chemicals.")

In [ ]:
master_ml_df.columns

In [ ]:
# Save your progress!
master_ml_df.to_csv(os.path.join(INTERIM_DIR, 'master_ml_df.csv'), index=False)

In [ ]:
import pandas as pd
import re

def categorize_termination_unified(reason):
    """
    Categorizes the termination reason by integrating audit_engine categories 
    with make_dataset negation logic and expanded keywords.
    """
    if pd.isnull(reason): 
        return 'Unknown'
    
    text = str(reason).lower()

    # 1. Define Negation Phrases
    negation_phrases = [
        'no safety concerns', 'no safety issues', 'not due to safety', 
        'benefit-risk', 'not for efficacy or safety', 'not for safety or efficacy',
        'not related to any efficacy or safety', 'not related to efficacy or safety',
        'no efficacy or safety issues', 'neither efficacy nor safety', 
        'neither safety nor efficacy', 'not due to any efficacy or safety', 
        'not due to efficacy or safety', 'not for reasons of efficacy or safety'
    ]
    negation_pattern = r'\b(?:' + '|'.join(negation_phrases) + r')\b'
    is_negated = re.search(negation_pattern, text) is not None

    # 2. Define Category Keywords
    # Efficacy: Merged keywords from both source scripts
    eff_keywords = [
        'efficacy', 'futility', 'benefit', 'endpoint', 'signal', 
        'lack of effect', 'superiority', 'insufficient signal'
    ]
    
    # Safety: Merged keywords from both source scripts
    safe_keywords = [
        'toxic', 'adverse event', 'safety', 'harm', 'risk', 
        'side effect', 'death', 'mortality', 'aes'
    ]
    
    # Accrual/Logistics: Merged keywords from both source scripts
    log_keywords = [
        'accrual', 'recruit', 'enroll', 'slow', 'low', 'insufficient', 
        'participant', 'recruitment', 'enrollment', 'funding', 'covid', 
        'personnel', 'feasibility', 'operational'
    ]
    
    # Business/Strategic: Keywords from audit_engine.py
    biz_keywords = [
        'business', 'strategic', 'funding', 'sponsor', 'priority', 
        'portfolio', 'commercial', 'budget'
    ]
    
    # Administrative: Keywords from audit_engine.py
    admin_keywords = [
        'administrative', 'operational', 'process', 'management'
    ]

    # 3. Application Logic (Respecting Negations)
    has_eff = any(kw in text for kw in eff_keywords)
    has_safe = any(kw in text for kw in safe_keywords)
    
    # Flags are only True if terms are present AND not negated
    eff_flag = has_eff and not is_negated
    safe_flag = has_safe and not is_negated

    # 4. Classification Hierarchy
    if safe_flag:
        return 'Safety'
    if eff_flag:
        return 'Efficacy'
    if any(kw in text for kw in biz_keywords):
        return 'Business/Strategic'
    if any(kw in text for kw in admin_keywords):
        return 'Administrative'
    if any(kw in text for kw in log_keywords):
        return 'Accrual/Logistics'
    
    # If safety/efficacy keywords were present but negated, it defaults to Strategic/Other logic
    if is_negated:
        return 'Business/Strategic'
        
    return 'Other/Unspecified'

# Apply the logic to your dataframe
master_ml_df['termination_category'] = master_ml_df['why_stopped'].apply(categorize_termination_unified)

In [ ]:
master_ml_df.columns

In [ ]:
import pandas as pd
import numpy as np

# 1. Apply the Unified Categorization Logic
print("Applying termination categorization logic...")
master_ml_df['termination_category'] = master_ml_df['why_stopped'].apply(categorize_termination_unified)

# 2. Check SMILES Availability
# We create a status column for easy distribution counting
master_ml_df['smiles_status'] = np.where(master_ml_df['smiles'].notna(), 'SMILES Available', 'SMILES Missing')

# 3. Generate the Distribution Reports
print("\n" + "="*40)
print("DISTRIBUTION: WHY_STOPPED CATEGORIES")
print("="*40)
termination_dist = master_ml_df['termination_category'].value_counts()
termination_pct = master_ml_df['termination_category'].value_counts(normalize=True) * 100

dist_summary = pd.DataFrame({
    'Count': termination_dist,
    'Percentage (%)': termination_pct.round(2)
})
print(dist_summary)

print("\n" + "="*40)
print("DISTRIBUTION: SMILES DATA AVAILABILITY")
print("="*40)
smiles_dist = master_ml_df['smiles_status'].value_counts()
smiles_pct = master_ml_df['smiles_status'].value_counts(normalize=True) * 100

smiles_summary = pd.DataFrame({
    'Count': smiles_dist,
    'Percentage (%)': smiles_pct.round(2)
})
print(smiles_summary)

# 4. Cross-tabulation: Categories vs SMILES
# This helps identify if certain failure types (like 'Unknown') are harder to enrich
print("\n" + "="*40)
print("CROSS-TAB: CATEGORY vs. SMILES AVAILABILITY")
print("="*40)
cross_tab = pd.crosstab(master_ml_df['termination_category'], master_ml_df['smiles_status'])
print(cross_tab)

In [ ]:
# Check only the categories we just created
print(master_ml_df['termination_category'].value_counts())

In [ ]:
import os

# This saves the clinical data and your new labels to a CSV
# This file is the input for the 'robust_pubchem_merge.py' script
master_ml_df.to_csv(os.path.join(INTERIM_DIR, 'master_ml_df.csv'), index=False)

print("Baseline saved successfully to data/interim/master_ml_df.csv")

In [ ]:
# Load the version created by the terminal script
master_ml_df = pd.read_csv(os.path.join(INTERIM_DIR, 'master_ml_df_enriched.csv'))

# Final verification of the molecular columns
print(f"Total rows loaded: {len(master_ml_df):,}")
print(f"SMILES found: {master_ml_df['smiles'].notna().sum():,}")
print(f"Molecular Weights found: {master_ml_df['molecular_weight'].notna().sum():,}")

In [ ]:
# Move the data from the 'notebook' (cache) to the 'spreadsheet' (dataframe)
master_ml_df['molecular_weight'] = master_ml_df['clean_name'].map(lambda x: results_cache.get(x, {}).get('MolecularWeight'))
master_ml_df['xlogp'] = master_ml_df['clean_name'].map(lambda x: results_cache.get(x, {}).get('XLogP'))
master_ml_df['smiles'] = master_ml_df['clean_name'].map(lambda x: results_cache.get(x, {}).get('CanonicalSMILES'))

# Now re-run your distribution check
master_ml_df['smiles_status'] = np.where(master_ml_df['smiles'].notna(), 'SMILES Available', 'SMILES Missing')
print(master_ml_df['smiles_status'].value_counts())

In [ ]:
import json
import os

# 1. IMMEDIATE SAFETY: Save the cache to a file so you never lose the 1.5 hours of work
try:
    with open('pubchem_cache_backup.json', 'w') as f:
        json.dump(results_cache, f)
    print("✅ SAFETY CHECK: Cache backed up to 'pubchem_cache_backup.json'. Your 1.5 hours of work is safe.")
except Exception as e:
    print(f"⚠️ Could not backup cache: {e}")

# 2. DIAGNOSTIC: Check why the mapping failed
cache_size = len(results_cache)
print(f"\nDIAGNOSTIC REPORT:")
print(f"Items in results_cache: {cache_size}")

if cache_size > 0:
    # Look at the first key in the cache and the first few names in the dataframe
    first_key = list(results_cache.keys())[0]
    first_name_in_df = master_ml_df['clean_name'].iloc[0]
    
    print(f"Sample Key in Cache: '{first_key}' (Type: {type(first_key)})")
    print(f"Sample Name in DF:   '{first_name_in_df}' (Type: {type(first_name_in_df)})")
    
    # Check for a specific known chemical
    test_chem = "Prednisone"
    if test_chem in results_cache:
        print(f"✅ Success: '{test_chem}' found in cache.")
    else:
        print(f"❌ Error: '{test_chem}' NOT found in cache. This suggests the cache might have been reset.")

# 3. RECOVERY: Brute Force Mapping (Handles string/numeric and whitespace issues)
print("\nPerforming Brute Force Mapping...")

# We convert both the key and the lookup value to strings and strip whitespace to ensure a match
def force_lookup(val, field):
    # Convert value to string and strip whitespace
    s_val = str(val).strip()
    # Try direct lookup, then try looking up in a string-ified version of the cache
    data = results_cache.get(val) or results_cache.get(s_val)
    if data:
        return data.get(field)
    return None

master_ml_df['molecular_weight'] = master_ml_df['clean_name'].apply(lambda x: force_lookup(x, 'MolecularWeight'))
master_ml_df['xlogp'] = master_ml_df['clean_name'].apply(lambda x: force_lookup(x, 'XLogP'))
master_ml_df['smiles'] = master_ml_df['clean_name'].apply(lambda x: force_lookup(x, 'CanonicalSMILES'))

# 4. FINAL VERIFICATION
master_ml_df['smiles_status'] = np.where(master_ml_df['smiles'].notna(), 'SMILES Available', 'SMILES Missing')
print("\nUPDATED DISTRIBUTION:")
print(master_ml_df['smiles_status'].value_counts())

In [ ]:
import pandas as pd
import numpy as np

# 1. Force master_ml_df to be a clean, independent copy to avoid "View" issues
master_ml_df = master_ml_df.copy()

# 2. Convert the Cache into a clean "Enrichment DataFrame"
enrichment_data = []
for name, data in results_cache.items():
    if data:
        enrichment_data.append({
            'lookup_name': str(name).strip().lower(),
            'pubchem_mw': data.get('MolecularWeight'),
            'pubchem_xlogp': data.get('XLogP'),
            'pubchem_smiles': data.get('CanonicalSMILES')
        })

enrichment_df = pd.DataFrame(enrichment_data)

# 3. Prepare master_ml_df for the join by creating a matching key
master_ml_df['lookup_name'] = master_ml_df['clean_name'].astype(str).str.strip().str.lower()

# 4. Perform the LEFT JOIN (The "Handshake")
# This replaces the unreliable .map() with a structural merge
master_ml_df = master_ml_df.merge(enrichment_df, on='lookup_name', how='left')

# 5. Cleanup: Move merged data to your original columns and drop the temp ones
master_ml_df['molecular_weight'] = master_ml_df['pubchem_mw']
master_ml_df['xlogp'] = master_ml_df['pubchem_xlogp']
master_ml_df['smiles'] = master_ml_df['pubchem_smiles']

master_ml_df.drop(columns=['lookup_name', 'pubchem_mw', 'pubchem_xlogp', 'pubchem_smiles'], inplace=True)

# 6. VERIFICATION
master_ml_df['smiles_status'] = np.where(master_ml_df['smiles'].notna(), 'SMILES Available', 'SMILES Missing')
found_count = master_ml_df[master_ml_df['smiles_status'] == 'SMILES Available'].shape[0]

print(f"✅ Recovery Complete!")
print(f"Total Rows in DF: {len(master_ml_df):,}")
print(f"Rows now enriched with SMILES: {found_count:,}")

# Show distribution of SMILES by your Audit Categories
print("\nEnrichment by Category:")
print(pd.crosstab(master_ml_df['termination_category'], master_ml_df['smiles_status']))

In [ ]:
len(master_ml_df)

In [ ]:
import os
print(f"Direct the agent to this path: {os.path.abspath(os.path.join(INTERIM_DIR, 'master_ml_df.csv'))}")

In [ ]:
master_ml_df.columns

In [ ]:
master_ml_df['smiles'].notna().sum()

In [ ]:
# Use the exact path you just copied
master_ml_df_enriched = pd.read_csv(r'C:\Users\Ownwer\Documents\BIFX\BIFX546\Final_Project\data\interim\master_ml_df_enriched.csv')

In [ ]:
len(master_ml_df_enriched)

In [ ]:
# 1. Check for the most duplicated Trial
top_duplicates = master_ml_df['nct_id'].value_counts().head(5)
print("Top Duplicated Trials:")
print(top_duplicates)

# 2. Inspect a single duplicated trial to see what columns are different
sample_nct = top_duplicates.index[0]
print(f"\nInspecting NCT ID: {sample_nct}")
display(master_ml_df[master_ml_df['nct_id'] == sample_nct].head(10))

In [ ]:
# Quick sanity check
print(f"Total rows loaded: {len(master_ml_df):,}")
print(f"SMILES found: {master_ml_df['smiles'].notna().sum():,}")

# Look at the first 5 drugs that have SMILES
print(master_ml_df[master_ml_df['smiles'].notna()][['clean_name', 'smiles']].head())

In [ ]:
master_ml_df['molecular_weight'].notna().sum()

In [ ]:
master_ml_df['xlogp'].notna().sum()